# Phase 5 — Build the Analytical Dataset

## Objective

The goal of this notebook is to build the analytical dataset required for the
Refresh / Content Opportunity Scoring research direction.

The analytical dataset will combine content-level characteristics with
historical search performance while preserving the intended unit of analysis.

The final analytical dataset will contain one row per content item.

## Task 5.1 — Select Relevant Tables

Based on the research question and the findings from the previous phases,
the initial analytical dataset will use two primary sources:

1. `dim_content`
   - Provides content-level characteristics.
   - Contains the primary content identifier.
   - Provides metadata that may be used as analytical features.

2. `fact_content_daily_performance_sample`
   - Provides historical daily search performance.
   - Contains GSC performance metrics such as impressions, clicks, and position.
   - Provides the time dimension required to construct historical performance features.

The following sources will not be included in the initial analytical dataset:

- `dim_clients`: client-level information is not required for the current
  content-focused research question.
- `fact_content_query_90d`: query-level data is not required for the initial
  content-level analytical dataset and may introduce additional aggregation
  complexity. It may be considered later if query-level signals are shown to
  be relevant.

## Task 5.2 — Define Unit of Analysis

### Objective

Define the unit of analysis for the analytical dataset before performing joins or aggregations.

The research question focuses on ranking content items by their priority for review based on historical search performance and content-level characteristics.

### Unit of Analysis

The primary unit of analysis is:

> **One content item for one client.**

Each final row will represent a unique combination of:

- `client_hash_id`
- `content_hash_id`

The daily performance table contains multiple records for the same content item across different report dates. Therefore, daily performance records must be aggregated to the content-item level before joining them with content-level characteristics.

### Expected Final Grain

The final analytical dataset should have:

`client_hash_id + content_hash_id`

as its unique key.

This means that each content item should appear only once per client.

### Grain Transformation

The analytical dataset will transform the data from:

`report_date + client_hash_id + content_hash_id`

to:

`client_hash_id + content_hash_id`

by aggregating historical performance metrics across the available reporting period.

### Why This Unit Was Chosen

This unit matches the research question because the final output is intended to rank content items by their priority for review or improvement.

Using the content item as the primary unit allows us to combine:

- historical search performance
- content-level characteristics
- content metadata

without creating multiple rows for the same content item.

### Important Consideration

The exact performance aggregation and time window will be determined during the analytical dataset and EDA stages rather than being assumed in advance.

In [7]:
import duckdb

con = duckdb.connect()

performance_file = 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'

performance_file

'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet'

In [11]:
from huggingface_hub import hf_hub_download

performance_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="fact_content_daily_performance_sample.parquet",
    repo_type="dataset"
)

print("Downloaded to:", performance_file)


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\fact_content_daily_performance_sample.parquet


In [12]:
performance_grain = con.execute("""
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM read_parquet(?)
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    ORDER BY row_count DESC
""", [performance_file]).df()

performance_grain.head(10)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count
0,2026-06-24,client_b77d0d5f08f05e64,content_41a620c650d2fea1,2
1,2026-06-20,client_1a730cb2640a1abf,content_42239c7e1161beee,2
2,2026-06-25,client_4a18d1793d92fb84,content_4753dfdc4e7ea44e,2
3,2026-06-21,client_b77d0d5f08f05e64,content_a0e4b6d90b08c770,2
4,2026-06-28,client_b77d0d5f08f05e64,content_f29d4fdf379f08d3,2
5,2026-06-25,client_b77d0d5f08f05e64,content_84105db6e8eb2343,2
6,2026-06-25,client_b77d0d5f08f05e64,content_b11af2c267f7170f,2
7,2026-06-25,client_b77d0d5f08f05e64,content_025a61a53d422411,2
8,2026-06-26,client_a2eeb8899886adde,content_97a0041b110ebc25,2
9,2026-06-22,client_1a730cb2640a1abf,content_4112607d223e829e,2


In [13]:
duplicate_groups = con.execute("""
    SELECT
        COUNT(*) AS duplicate_groups,
        SUM(row_count - 1) AS extra_rows
    FROM (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            COUNT(*) AS row_count
        FROM read_parquet(?)
        GROUP BY
            report_date,
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) > 1
    )
""", [performance_file]).df()

duplicate_groups

,duplicate_groups,extra_rows
0,6390,6390.0


In [14]:
content_client_grain = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id) AS unique_clients,
        COUNT(DISTINCT content_hash_id) AS unique_content_ids,
        COUNT(DISTINCT
            client_hash_id || '|' || content_hash_id
        ) AS unique_client_content_pairs
    FROM read_parquet(?)
""", [performance_file]).df()

content_client_grain

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_clients,unique_content_ids,unique_client_content_pairs
0,11694072,65,409205,409205


In [15]:
target_grain_check = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS daily_rows
    FROM read_parquet(?)
    GROUP BY
        client_hash_id,
        content_hash_id
    ORDER BY daily_rows DESC
    LIMIT 10
""", [performance_file]).df()

target_grain_check

,client_hash_id,content_hash_id,daily_rows
0,client_b77d0d5f08f05e64,content_f29d4fdf379f08d3,36
1,client_aef6ffea193da149,content_b0db72093df5ecb0,36
2,client_1a8bf67cad4ee525,content_628f314bdd075042,36
3,client_06d356715a8ff3b6,content_2b4f82281f289e87,36
4,client_06d356715a8ff3b6,content_665d40d240c948cd,36
5,client_def0955f7a377868,content_c52c3798a1412de7,36
6,client_06d356715a8ff3b6,content_03a8b5950a52518a,36
7,client_a22068e339bf95f5,content_6e297832147a30e6,36
8,client_aef6ffea193da149,content_fa6976b22bbe7e4a,36
9,client_b77d0d5f08f05e64,content_67927a2aa00a05a7,36


### Task 5.2 Conclusion

The unit of analysis is defined as one content item for one client.

The daily performance table contains 11,694,072 rows across 65 clients and 409,205 unique content items.

The number of unique `client_hash_id + content_hash_id` combinations is also 409,205, confirming that the content items in the performance data are uniquely associated with a client.

The current performance grain is:

`report_date + client_hash_id + content_hash_id`

while the target analytical grain is:

`client_hash_id + content_hash_id`

Multiple daily records exist for the same content item because performance is recorded over time. Therefore, daily performance metrics must be aggregated to the content-client level before joining them with content-level characteristics.

The performance data also contains 6,390 duplicate groups identified during the data quality phase. These duplicates will be handled during analytical dataset construction in Task 5.3.

The final analytical dataset should contain one row per content item for each client.

## Task 5.3 — Build Analytical Dataset

### Objective

Build a reproducible analytical dataset at the content-client level by combining:

- Content-level characteristics from `dim_content`
- Historical search performance from `fact_content_daily_performance_sample`

The final dataset will contain one row per:

`client_hash_id + content_hash_id`

The construction process will preserve the defined grain and validate row counts, joins, and duplicate keys.

In [24]:
from huggingface_hub import hf_hub_download

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

print("Downloaded to:", content_file)


Downloaded to: C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\dim_content.parquet


In [25]:
content_file


'C:\\Users\\anasm\\.cache\\huggingface\\hub\\datasets--FlyRank--internship-warehouse\\snapshots\\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\\dim_content.parquet'

In [27]:
content_count = con.execute("""
    SELECT COUNT(*) AS row_count
    FROM read_parquet(?)
""", [content_file]).df()

content_count


,row_count
0,519606


### 5.3.1 — Remove Exact Duplicate Performance Rows

The raw performance table contains exact duplicate rows identified during the data quality phase.

These duplicates are removed only from the analytical copy of the data. The original raw dataset remains unchanged.

Removing exact duplicates prevents duplicated observations from artificially increasing aggregated performance metrics.

In [28]:
performance_deduplicated = con.execute("""
    SELECT DISTINCT *
    FROM read_parquet(?)
""", [performance_file]).df()

performance_deduplicated.shape

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(11687682, 31)

In [29]:
original_count = con.execute("""
    SELECT COUNT(*) AS row_count
    FROM read_parquet(?)
""", [performance_file]).fetchone()[0]

deduplicated_count = len(performance_deduplicated)

print("Original rows:", original_count)
print("Deduplicated rows:", deduplicated_count)
print("Removed rows:", original_count - deduplicated_count)

Original rows: 11694072
Deduplicated rows: 11687682
Removed rows: 6390


In [32]:
performance_file


'C:\\Users\\anasm\\.cache\\huggingface\\hub\\datasets--FlyRank--internship-warehouse\\snapshots\\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\\fact_content_daily_performance_sample.parquet'

In [33]:
con.execute(f"""
    CREATE OR REPLACE VIEW performance_deduplicated AS
    SELECT DISTINCT *
    FROM read_parquet('{performance_file}')
""")


In [34]:
con.execute("""
    SELECT *
    FROM performance_deduplicated
    LIMIT 10
""").df()


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-01,client_62f4a7e64f5e0096,content_c51eccaa3eee3c77,True,False,True,<NA>,62,0,482,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
1,2026-06-01,client_62f4a7e64f5e0096,content_ff626dff46ecb83e,True,False,True,<NA>,17,0,123,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
2,2026-06-01,client_62f4a7e64f5e0096,content_66e46fd8ae829d7a,True,False,True,<NA>,164,0,1665,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
3,2026-06-01,client_62f4a7e64f5e0096,content_849a73af52679d0e,True,False,True,<NA>,36,0,869,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
4,2026-06-01,client_62f4a7e64f5e0096,content_c54fa47e2a04538d,True,False,True,<NA>,12,0,390,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
5,2026-06-01,client_62f4a7e64f5e0096,content_0814cec34c2671e4,True,False,True,<NA>,3,0,24,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
6,2026-06-01,client_62f4a7e64f5e0096,content_c8bb8248b3735e02,True,False,False,<NA>,0,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
7,2026-06-01,client_62f4a7e64f5e0096,content_bc225b4b5ce28bec,True,False,True,<NA>,45,1,574,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
8,2026-06-01,client_62f4a7e64f5e0096,content_52f2e0298d8fc7dd,True,False,True,<NA>,9,0,172,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06
9,2026-06-01,client_62f4a7e64f5e0096,content_7925b37e28c2e37a,True,False,True,<NA>,2,0,70,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-06


In [35]:
con.execute("""
    SELECT COUNT(*) AS row_count
    FROM performance_deduplicated
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count
0,11687682


### 5.3.2 — Aggregate Historical Performance

The daily performance data is aggregated from the daily level to the content-client level.

The aggregation is performed after removing exact duplicate rows.

At this stage, the goal is to preserve historical performance information while producing one performance record for each content item and client.

No specific historical time window is imposed at this stage. The available reporting period is retained so that the appropriate analytical window can be evaluated during EDA.

In [37]:
con.execute("PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp/duckdb_tmp';")


In [39]:
con.execute("""
    CREATE OR REPLACE VIEW performance_aggregated AS
    SELECT
        client_hash_id,
        content_hash_id,

        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS reporting_days,

        SUM(gsc_impressions) AS total_gsc_impressions,
        SUM(gsc_clicks) AS total_gsc_clicks,

        SUM(gsc_sum_position) AS total_gsc_sum_position,

        AVG(gsc_avg_position) AS mean_gsc_avg_position

    FROM performance_deduplicated

    GROUP BY
        client_hash_id,
        content_hash_id
""")


In [42]:
con.execute("PRAGMA memory_limit='1GB';")
con.execute("PRAGMA temp_directory='C:/Users/anasm/AppData/Local/Temp';")
con.execute("DROP VIEW IF EXISTS performance_aggregated;")


In [43]:
con.execute("""
    CREATE OR REPLACE TABLE performance_aggregated AS
    SELECT
        client_hash_id,
        content_hash_id,

        MIN(report_date) AS first_report_date,
        MAX(report_date) AS last_report_date,
        COUNT(DISTINCT report_date) AS reporting_days,

        SUM(gsc_impressions) AS total_gsc_impressions,
        SUM(gsc_clicks) AS total_gsc_clicks,
        SUM(gsc_sum_position) AS total_gsc_sum_position,
        AVG(gsc_avg_position) AS mean_gsc_avg_position

    FROM performance_deduplicated

    GROUP BY
        client_hash_id,
        content_hash_id
""")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [44]:
con.execute("""
    SELECT *
    FROM performance_aggregated
    LIMIT 10
""").df()


,client_hash_id,content_hash_id,first_report_date,last_report_date,reporting_days,total_gsc_impressions,total_gsc_clicks,total_gsc_sum_position,mean_gsc_avg_position
0,client_73cda7b4e4f265ea,content_042515eda943fdc3,2026-06-01,2026-06-30,30,47.0,0.0,1822.0,36.048611
1,client_73cda7b4e4f265ea,content_9a0ba88b288506e1,2026-06-01,2026-06-30,30,39.0,0.0,459.0,10.500000
2,client_73cda7b4e4f265ea,content_78e7a3bb567c786c,2026-06-01,2026-06-30,30,5.0,0.0,374.0,72.875000
3,client_73cda7b4e4f265ea,content_07dc3b7c0fa80365,2026-06-01,2026-06-30,30,722.0,0.0,10430.0,16.332035
4,client_73cda7b4e4f265ea,content_9cdf69b727398b26,2026-06-01,2026-06-30,30,12.0,0.0,504.0,47.958333
5,client_73cda7b4e4f265ea,content_36bd68e779ad2527,2026-06-01,2026-06-30,30,583.0,0.0,4351.0,7.429495
6,client_73cda7b4e4f265ea,content_a628544c06b8e9eb,2026-06-01,2026-06-30,30,814.0,0.0,8049.0,11.526799
7,client_73cda7b4e4f265ea,content_4ad1bcbde1e8e854,2026-06-01,2026-06-30,30,4228.0,23.0,28602.0,7.009246
8,client_73cda7b4e4f265ea,content_7d5a30301b31d23b,2026-06-01,2026-06-30,30,1252.0,3.0,21861.0,31.084124
9,client_73cda7b4e4f265ea,content_886f0e41955682e5,2026-06-01,2026-06-30,30,9.0,0.0,244.0,23.333333


In [45]:
grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
            AS unique_client_content_pairs
    FROM performance_aggregated
""").df()

grain_check

,total_rows,unique_client_content_pairs
0,409205,409205


In [46]:
duplicate_final_grain = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM performance_aggregated
    GROUP BY
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

duplicate_final_grain

,client_hash_id,content_hash_id,row_count


### Aggregation Result

The daily performance data was successfully transformed to the target content-client grain.

After exact duplicate removal, the daily records were aggregated using:

- `MIN(report_date)` to identify the first available reporting date.
- `MAX(report_date)` to identify the last available reporting date.
- `COUNT(DISTINCT report_date)` to measure reporting coverage.
- `SUM(gsc_impressions)` to calculate total impressions.
- `SUM(gsc_clicks)` to calculate total clicks.
- `SUM(gsc_sum_position)` to retain the aggregated position signal.
- `AVG(gsc_avg_position)` to calculate the mean reported average position.

The aggregation is stored as a DuckDB table rather than being loaded entirely into pandas, because the source performance dataset contains more than 11 million rows.

The resulting table is expected to contain one row per:

`client_hash_id + content_hash_id`

### 5.3.3 — Prepare Content-Level Dataset

The `dim_content` table provides content-level characteristics that complement the historical search performance data.

Only columns relevant to the current research direction are selected.

The content-level dataset must preserve the target grain:

`client_hash_id + content_hash_id`

before joining it with the aggregated performance data.

In [49]:
from huggingface_hub import hf_hub_download

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

print("content_file =", content_file)


content_file = C:\Users\anasm\.cache\huggingface\hub\datasets--FlyRank--internship-warehouse\snapshots\50cbf7c3909d07be4d1b5906b4d09e882e5acbf2\dim_content.parquet


In [50]:
content_count = con.execute(f"""
    SELECT COUNT(*) AS row_count
    FROM read_parquet('{content_file}')
""").df()

content_count


,row_count
0,519606


In [54]:
from huggingface_hub import hf_hub_download

content_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename="dim_content.parquet",
    repo_type="dataset"
)

con.execute(f"""
    CREATE OR REPLACE TABLE content_selected AS
    SELECT
        client_hash_id,
        content_hash_id,
        content_type,
        search_volume,
        competition,
        competition_level,
        cpc,
        main_intent,
        backlinks,
        category_count,
        char_count,
        word_count,
        last_optimized_date,
        optimization_eligible_date,
        is_published,
        is_deleted
    FROM read_parquet('{content_file}')
""")


In [56]:
con.execute("""
    SELECT *
    FROM content_selected
    LIMIT 10
""").df()


,client_hash_id,content_hash_id,content_type,search_volume,competition,competition_level,cpc,main_intent,backlinks,category_count,char_count,word_count,last_optimized_date,optimization_eligible_date,is_published,is_deleted
0,client_04660893ae39614a,content_004de9653278b5a4,keyword article,30,0.91,HIGH,0.98,transactional,16,3,15682,2555,NaT,NaT,True,False
1,client_04660893ae39614a,content_00dc5efae381b2ab,keyword article,10,0.00,LOW,0.00,commercial,0,4,15438,2430,NaT,NaT,True,False
2,client_04660893ae39614a,content_01410f2556c327ac,keyword article,480,0.36,MEDIUM,0.62,informational,169,4,16576,2645,NaT,NaT,True,False
3,client_04660893ae39614a,content_019f27f634053ca7,keyword article,0,0.00,LOW,0.00,transactional,0,4,15457,2522,NaT,NaT,True,False
4,client_04660893ae39614a,content_01efa71faea45dcc,keyword article,2400,0.70,HIGH,0.90,transactional,52,4,15776,2552,NaT,NaT,True,False
5,client_04660893ae39614a,content_01fc9e2e57898b55,keyword article,260,0.14,LOW,0.10,transactional,14,4,16387,2672,NaT,NaT,True,False
6,client_04660893ae39614a,content_0212158fa61c5fcb,keyword article,10,0.00,LOW,0.00,informational,0,4,14383,2396,NaT,NaT,True,False
7,client_04660893ae39614a,content_023d807c7d922db1,keyword article,10,0.57,MEDIUM,0.00,informational,0,4,11513,1929,NaT,NaT,True,False
8,client_04660893ae39614a,content_026f7405cc253242,keyword article,4400,0.80,HIGH,1.70,transactional,33,4,15296,2481,NaT,NaT,True,False
9,client_04660893ae39614a,content_02c8b23ea5bdb275,keyword article,390,0.45,MEDIUM,1.36,transactional,33,4,14576,2357,NaT,NaT,True,False


In [55]:
content_grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
            AS unique_client_content_pairs
    FROM content_selected
""").df()

content_grain_check

,total_rows,unique_client_content_pairs
0,519606,519606


In [57]:
content_grain_check = con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
            AS unique_client_content_pairs
    FROM content_selected
""").df()

content_grain_check

,total_rows,unique_client_content_pairs
0,519606,519606


In [58]:
content_duplicates = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM content_selected
    GROUP BY
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

content_duplicates

,client_hash_id,content_hash_id,row_count


In [59]:
join_coverage = con.execute("""
    SELECT
        COUNT(*) AS performance_rows,
        COUNT(c.content_hash_id) AS matched_content_rows,
        COUNT(*) - COUNT(c.content_hash_id) AS unmatched_content_rows
    FROM performance_aggregated p
    LEFT JOIN content_selected c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
""").df()

join_coverage

,performance_rows,matched_content_rows,unmatched_content_rows
0,409205,409205,0


### 5.3.4 — Build Final Analytical Dataset

The aggregated performance table is joined with the selected content-level characteristics using:

`client_hash_id + content_hash_id`

A `LEFT JOIN` from the performance table is used so that content items with valid historical performance are retained even when some content-level attributes are missing.

The resulting dataset represents the analytical starting point for EDA and feature engineering.

In [60]:
con.execute("""
    CREATE OR REPLACE TABLE analytical_dataset AS
    SELECT
        p.client_hash_id,
        p.content_hash_id,

        p.first_report_date,
        p.last_report_date,
        p.reporting_days,

        p.total_gsc_impressions,
        p.total_gsc_clicks,
        p.total_gsc_sum_position,
        p.mean_gsc_avg_position,

        c.content_type,
        c.search_volume,
        c.competition,
        c.competition_level,
        c.cpc,
        c.main_intent,
        c.backlinks,
        c.category_count,
        c.char_count,
        c.word_count,
        c.last_optimized_date,
        c.optimization_eligible_date,
        c.is_published,
        c.is_deleted

    FROM performance_aggregated p

    LEFT JOIN content_selected c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
""")

In [ ]:
final_count_check = con.execute("""
    SELECT
        COUNT(*) AS final_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id)
            AS unique_client_content_pairs
    FROM analytical_dataset
""").df()

final_count_check

,final_rows,unique_client_content_pairs
0,409205,409205


In [ ]:
final_duplicates = con.execute("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS row_count
    FROM analytical_dataset
    GROUP BY
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()

final_duplicates

,client_hash_id,content_hash_id,row_count


In [ ]:
final_missingness = con.execute("""
    SELECT
        COUNT(*) AS total_rows,

        COUNT(*) FILTER (
            WHERE search_volume IS NULL
        ) AS missing_search_volume,

        COUNT(*) FILTER (
            WHERE main_intent IS NULL
        ) AS missing_main_intent,

        COUNT(*) FILTER (
            WHERE char_count IS NULL
        ) AS missing_char_count,

        COUNT(*) FILTER (
            WHERE word_count IS NULL
        ) AS missing_word_count,

        COUNT(*) FILTER (
            WHERE last_optimized_date IS NULL
        ) AS missing_last_optimized_date,

        COUNT(*) FILTER (
            WHERE optimization_eligible_date IS NULL
        ) AS missing_optimization_eligible_date,

        COUNT(*) FILTER (
            WHERE mean_gsc_avg_position IS NULL
        ) AS missing_mean_gsc_avg_position

    FROM analytical_dataset
""").df()

final_missingness

,total_rows,missing_search_volume,missing_main_intent,missing_char_count,missing_word_count,missing_last_optimized_date,missing_optimization_eligible_date,missing_mean_gsc_avg_position
0,409205,66001,71511,107758,107758,363809,363809,200569


### Task 5.3 Conclusion

The analytical dataset was successfully constructed at the content-client level.

The construction process:

1. Removed exact duplicate performance rows from the analytical copy.
2. Aggregated daily performance to the `client_hash_id + content_hash_id` grain.
3. Selected relevant content-level characteristics from `dim_content`.
4. Joined the aggregated performance data with content-level characteristics.
5. Validated the final row count and analytical grain.
6. Checked join coverage and missing values.

The final dataset is designed to support the next phases of the project, including exploratory data analysis, feature engineering, baseline construction, and opportunity scoring.

Missing values are retained at this stage and will be evaluated during EDA and feature engineering rather than being removed automatically.

In [64]:
join_coverage = con.execute("""
    SELECT
        COUNT(*) AS performance_rows,
        COUNT(c.content_hash_id) AS matched_content_rows,
        COUNT(*) - COUNT(c.content_hash_id) AS unmatched_content_rows
    FROM performance_aggregated p
    LEFT JOIN content_selected c
        ON p.client_hash_id = c.client_hash_id
        AND p.content_hash_id = c.content_hash_id
""").df()

join_coverage

,performance_rows,matched_content_rows,unmatched_content_rows
0,409205,409205,0


## Task 5.3 — Build Analytical Dataset

The analytical dataset was successfully built by aggregating daily content performance to the `client_hash_id + content_hash_id` level and joining the selected content-level characteristics.

Exact duplicate performance records were removed from the analytical copy before aggregation. The resulting performance data was aggregated by content-client pair, preserving the intended unit of analysis.

The final join achieved complete coverage: all 409,205 performance content-client pairs were matched with a corresponding record in `dim_content`.

The final analytical dataset contains 409,205 rows and 409,205 unique client-content pairs, confirming that the intended grain was preserved and no row multiplication occurred during the join.

Missing values were retained rather than removed at this stage. Their distribution will be investigated during the EDA and handled according to the role and meaning of each feature.

Therefore, the analytical dataset is ready for the next phase: Exploratory Data Analysis (EDA).